In [5]:
!pip install hdfs


In [6]:
from hdfs import InsecureClient

client = InsecureClient('http://namenode:9870', user='root')
client

<InsecureClient(url='http://namenode:9870')>

In [7]:
dirs = ['/datalake/renfe/raw/estaciones', '/datalake/renfe/raw/viajeros', '/datalake/renfe/procesed', '/datalake/renfe/logs']

for dir in dirs:
    client.makedirs(dir, 755)
    print(f"Se ha creado el directorio {dir}")
    
print(f"Elementos en /: {client.list('/datalake/renfe')}")

Se ha creado el directorio /datalake/renfe/raw/estaciones
Se ha creado el directorio /datalake/renfe/raw/viajeros
Se ha creado el directorio /datalake/renfe/procesed
Se ha creado el directorio /datalake/renfe/logs
Elementos en /: ['logs', 'procesed', 'raw']


In [8]:
archivos = ['estaciones_completo.csv', 'estaciones_completo.csv']

for archivo in archivos:
    client.upload(
        hdfs_path=dirs[0],
        local_path=archivo,
        overwrite=True,
        n_threads=4
    )
    print(f'Archivo {archivo} subido correctamente a {dirs[0]}')
    
archivo='viajeros_tramo.csv'

client.upload(
    hdfs_path=dirs[1],
    local_path=archivo,
    overwrite=True,
    n_threads=4
)
print(f'Archivo {archivo} subido correctamente a {dirs[1]}')


archivos_subidos = client.list(dirs[0], status=True)
for archivo_subido, metadatos in archivos_subidos:
    print(f'Nombre:{archivo_subido}')
    print(f'Tamaño: {metadatos["length"]}')
    print(f'Permisos: {metadatos["permission"]}')
    print(f'Factor de replicacion: {metadatos["replication"]}')

Archivo estaciones_completo.csv subido correctamente a /datalake/renfe/raw/estaciones
Archivo estaciones_completo.csv subido correctamente a /datalake/renfe/raw/estaciones
Archivo viajeros_tramo.csv subido correctamente a /datalake/renfe/raw/viajeros
Nombre:estaciones_completo.csv
Tamaño: 142381
Permisos: 644
Factor de replicacion: 3


In [15]:
import csv

IDX_PROVINCIA = 7
total_estaciones_filtradas = 0
PROVINCIA_OBJETIVO = "LEÓN"
with client.read(dirs[0] + '/estaciones_completo.csv', encoding='latin-1') as reader:
    with client.write(dirs[2] + '/estaciones_filtradas.csv', overwrite=True, encoding='utf-8') as writer:
        cabecera = reader.readline()
        writer.write(cabecera)

        for linea in reader:
            if not linea.strip():
                continue
            fila = next(csv.reader([linea], delimiter=';', quotechar='"'))

            if len(fila) > IDX_PROVINCIA:
                provincia_actual = fila[IDX_PROVINCIA]
                if PROVINCIA_OBJETIVO == provincia_actual:
                    writer.write(linea)
                    total_estaciones_filtradas +=1
                
with client.read(dirs[2] + '/estaciones_filtradas.csv', encoding='utf-8') as reader:
    for i in range(5):
        print (reader.readline().strip())

"CODIGO";"DESCRIPCION";"LATITUD";"LONGITUD";"DIRECION";"CP";"POBLACION";"PROVINCIA";"PAIS";"CERCANIAS";"ANCHO MÉTRICO";"COMUN"
"05755";"LA LLAMA DE LA GUZPEÑA";"42.7702481";"-5.0412478";"Ctra. Cl - 626 --- Estación Ferrocarril, S/N";"24893";"PRADO DE LA GUZPEÑA";"LEÓN";"ESPAÑA";"SI";"SI";""
"05756";"LA ESPINA";"42.7868560";"-4.9052025";"Ctra. Cl - 626 --- Estación Ferrocarril, S/N";"24889";"VALDERRUEDA";"LEÓN";"ESPAÑA";"SI";"SI";""
"05757";"VALCUENDE";"42.7655780";"-4.9292558";"C/ Arriba, S/N";"24889";"VALDERRUEDA";"LEÓN";"ESPAÑA";"SI";"SI";""
"05758";"PUENTE ALMUHEY";"42.7876320";"-4.9718992";"Ctra. General  S/N- Puente Almuhey";"24880";"VALDERRUEDA";"LEÓN";"ESPAÑA";"SI";"SI";""


In [25]:
with client.write(dirs[3]+ '/ingesta_log.csv', encoding='utf-8', overwrite=True) as writer:
    writer.write('[PROCESADO] Se han filtrado un total de X estaciones para la provincia seleccionada.\n')
with client.write(dirs[3]+'/ingesta_log.csv', encoding='utf8', append=True) as writer:
    writer.write(f"[PROCESADO] Se han filtrado un total de {total_estaciones_filtradas} ""estaciones para la provincia seleccionada.")
        
with client.read(dirs[3]+'/ingesta_log.csv', encoding='utf-8') as reader:
    print(reader.read())

[PROCESADO] Se han filtrado un total de X estaciones para la provincia seleccionada.
[PROCESADO] Se han filtrado un total de 43 estaciones para la provincia seleccionada.


In [ ]:
# FASE 5: Exportación y limpieza

# 1. Descargar archivo procesado
client.download(
    dirs[2] + '/estaciones_filtradas.csv',
    'entrega_estaciones.csv',
    overwrite=True
)

print("Archivo descargado correctamente")

client.delete(dirs[0] + '/estaciones_feve.csv')
archivos = client.list(dirs[0])
print("Archivos restantes:", archivos)

assert 'estaciones_completo.csv' in archivos
print("El archivo estaciones_completo.csv permanece intacto.")

def reset_datalake():
    client.delete('/datalake/renfe', recursive=True)

Archivo descargado correctamente
Archivos restantes: ['estaciones_completo.csv']
El archivo estaciones_completo.csv permanece intacto.


**⬅️ [Volver atras](https://luistona.github.io/bda_2026_lalg/ut02/pr0204/pr0204.html)**